In [1]:
from statistics import mode
from PIL import Image, ImageDraw
import xgoscreen.LCD_2inch as LCD_2inch

import cv2
from keras.models import load_model
import numpy as np

from utils.datasets import get_labels
from utils.inference import detect_faces
from utils.inference import draw_text
from utils.inference import draw_bounding_box
from utils.inference import apply_offsets
from utils.inference import load_detection_model
from utils.preprocessor import preprocess_input

In [2]:
splash_theme_color = (255,255,255)
mydisplay = LCD_2inch.LCD_2inch()
mydisplay.Init()
mydisplay.clear()
# Init Splash
splash = Image.new("RGB", (mydisplay.height, mydisplay.width), splash_theme_color)
draw = ImageDraw.Draw(splash)
mydisplay.ShowImage(splash)

Screen already initialized.


In [3]:
import ipywidgets.widgets as widgets
from IPython.display import display

#显示摄像头组件 Display camera components
image_widget = widgets.Image(format='jpeg', width=320, height=240)

#bgr8转jpeg格式  bgr8 to jpeg format
import enum
def bgr8_to_jpeg(value, quality=75):
    return bytes(cv2.imencode('.jpg', value)[1])


In [4]:
# parameters for loading data and images
detection_model_path = '/home/pi/RaspberryPi-CM5/face_classification-master/trained_models/detection_models/haarcascade_frontalface_default.xml'
emotion_model_path = '/home/pi/RaspberryPi-CM5/face_classification-master/trained_models/emotion_models/fer2013_mini_XCEPTION.102-0.66.hdf5'
emotion_labels = get_labels('fer2013')
# hyper-parameters for bounding boxes shape
frame_window = 10
emotion_offsets = (20, 40)

# loading models
face_detection = load_detection_model(detection_model_path)
emotion_classifier = load_model(emotion_model_path, compile=False)

# getting input model shapes for inference
emotion_target_size = emotion_classifier.input_shape[1:3]

# starting lists for calculating modes
emotion_window = []

In [5]:
# starting video streaming
from key import Button,language
from picamera2 import Picamera2
picam2 = Picamera2()
picam2.configure(picam2.create_preview_configuration(main={"format": 'RGB888', "size": (320, 240)}))
picam2.start()
#print("摄像头初始化完毕")
button = Button()
la = language()

/home/pi/DOGZILLA_Lite_class/5.AI Visual Recognition Course/19. Emotion_Recognition
/home/pi/RaspberryPi-CM5/language/language.ini
cn


[1:01:43.280323803] [457964]  INFO Camera camera_manager.cpp:325 libcamera v0.3.2+99-1230f78d
[1:01:43.287639879] [458205]  INFO RPI pisp.cpp:695 libpisp version v1.0.7 28196ed6edcf 29-08-2024 (16:33:32)
[1:01:43.297564964] [458205]  INFO RPI pisp.cpp:1154 Registered camera /base/axi/pcie@120000/rp1/i2c@88000/ov5647@36 to CFE device /dev/media2 and ISP device /dev/media0 using PiSP variant BCM2712_D0
[1:01:43.301260511] [457964]  INFO Camera camera.cpp:1197 configuring streams: (0) 320x240-RGB888 (1) 640x480-GBRG_PISP_COMP1
[1:01:43.301366789] [458205]  INFO RPI pisp.cpp:1450 Sensor: /base/axi/pcie@120000/rp1/i2c@88000/ov5647@36 - Selected sensor format: 640x480-SGBRG10_1X10 - Selected CFE format: 640x480-PC1g


In [6]:
display(image_widget)
Count = 10
Count_num = 0
last_emotion = None
while True:
    bgr_image = picam2.capture_array()
    gray_image = cv2.cvtColor(bgr_image, cv2.COLOR_BGR2GRAY)
    rgb_image = cv2.cvtColor(bgr_image, cv2.COLOR_BGR2RGB)
    faces = detect_faces(face_detection, gray_image)

    for face_coordinates in faces:

        x1, x2, y1, y2 = apply_offsets(face_coordinates, emotion_offsets)
        gray_face = gray_image[y1:y2, x1:x2]
        try:
            gray_face = cv2.resize(gray_face, (emotion_target_size))
        except:
            continue

        gray_face = preprocess_input(gray_face, True)
        gray_face = np.expand_dims(gray_face, 0)
        gray_face = np.expand_dims(gray_face, -1)
        emotion_prediction = emotion_classifier.predict(gray_face)
        emotion_probability = np.max(emotion_prediction)
        emotion_label_arg = np.argmax(emotion_prediction)
        emotion_text = emotion_labels[emotion_label_arg]
        emotion_window.append(emotion_text)
        if emotion_text != last_emotion:
            Count_num = 0
        else:
            Count_num += 1
        last_emotion = emotion_text

        if len(emotion_window) > frame_window:
            emotion_window.pop(0)
        try:
            emotion_mode = mode(emotion_window)
        except:
            continue

        #print(f"当前的情绪为: {emotion_text},概率为{emotion_probability},连续计数为: {Count_num}")
        if la == "cn":
            print(f"当前的情绪为: {emotion_text},概率为{emotion_probability}")
        else:
            print(f"The current emotion is: {emotion_text},The probability is{emotion_probability}")
            
        if emotion_text == 'angry' and Count_num >= Count :
            color = emotion_probability * np.asarray((255, 0, 0))
            Count_num = 0
        elif emotion_text == 'sad' and Count_num >= Count :
            color = emotion_probability * np.asarray((0, 0, 255))
            Count_num = 0
        elif emotion_text == 'happy' and Count_num >= Count :
            color = emotion_probability * np.asarray((255, 255, 0))
            Count_num = 0
        elif emotion_text == 'surprise' and Count_num >= Count :
            color = emotion_probability * np.asarray((0, 255, 255))
            Count_num = 0
        elif emotion_text == 'neutral' and Count_num >= Count :
            color = emotion_probability * np.asarray((0, 255, 255))
            Count_num = 0
        else:
            color = emotion_probability * np.asarray((0, 255, 0))
            # # 假设其他情绪为恐惧和厌恶 Assuming other emotions are fear and disgust
            # if emotion_text in ['fear', 'scared']:  
            #     show("Stun", 8, "fear")  # 恐惧 fear
            # elif emotion_text in ['disgust', 'repulsed']:  
            #     show("shame", 11, "disgust")  # 厌恶 disgust

        color = color.astype(int)
        color = color.tolist()
        
        draw_bounding_box(face_coordinates, rgb_image, color)
        draw_text(face_coordinates, rgb_image, emotion_mode,
                  color, 0, -45, 1, 1)

    bgr_image = cv2.cvtColor(rgb_image, cv2.COLOR_RGB2BGR)
    
    image_widget.value = bgr8_to_jpeg(bgr_image)
    
    b, g, r = cv2.split(bgr_image)
    img = cv2.merge((r, g, b))
    imgok = Image.fromarray(img)
    mydisplay.ShowImage(imgok)  
    
    
    #cv2.imshow("frame",bgr_image)
    
    if button.press_b():
        break
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break


Image(value=b'', format='jpeg', height='240', width='320')

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 736ms/step
当前的情绪为: sad,概率为0.5327380895614624
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
当前的情绪为: sad,概率为0.6234347224235535
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
当前的情绪为: sad,概率为0.7869844436645508
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
当前的情绪为: sad,概率为0.722434937953949
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
当前的情绪为: sad,概率为0.7476294040679932
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
当前的情绪为: sad,概率为0.7534512281417847
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
当前的情绪为: sad,概率为0.7961926460266113
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
当前的情绪为: sad,概率为0.8065792322158813
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
当前的情绪为: sad,概率为0.6659350395202637
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
当前的情绪为: sad,概率为0.4512459337711334
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
当前的情绪为: sad,概率为0.8842028975486755
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
当前的情绪为: sad,概率为0.5209969878196716
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
当前的情绪为: sad,概率为0.5981606245040894
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
当前的情绪为: sad,概率为0.623762130

KeyboardInterrupt: 